# Day 38 / 42: LLM Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week5_nlp_llms/day38_llm_evaluation/day38_notebook.ipynb)

**#42DaysOfML**

---

## What You'll Learn

- Why evaluating LLMs is fundamentally harder than evaluating traditional ML models
- Traditional NLP metrics (ROUGE) and where they fall short for generative text
- How to build a **faithfulness** check: does the answer stay grounded in the source context?
- How to build an **answer relevancy** check: does the answer actually address the question?
- The **LLM-as-judge** pattern used in production evaluation pipelines
- A hands-on evaluation exercise on 5 real RAG-style answers, including 2 hallucinations you have to catch

This notebook uses lightweight, dependency-light implementations (TF-IDF cosine similarity, ROUGE) so every cell
runs offline with no API key required. The LLM-as-judge section shows you the real production pattern and how to
wire in an API key when you have one — it degrades gracefully without one.

**Reference:** The concepts here follow the evaluation framework described in Chip Huyen's *AI Engineering*
(chapter on evaluation methodology) and the metric definitions used by the [Ragas](https://github.com/explodinggradients/ragas)
library, one of the most widely used open-source RAG evaluation tools in production teams today.


## Setup

Everything here runs with libraries you already have from earlier days (`scikit-learn`), plus `rouge-score`
for one traditional metric comparison.


In [ ]:
!pip install rouge-score --quiet
print("Setup complete.")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer

pd.set_option('display.max_colwidth', None)
print("Libraries loaded.")

## 1. Why LLM Evaluation Is a Different Problem

In traditional ML, evaluation is mostly settled: you have a ground truth label and a metric like accuracy,
F1, or RMSE. There's one correct answer and a clean way to check against it.

LLMs break this in three specific ways:

1. **There's no single correct answer.** Two completely different sentences can both be a "correct" answer
   to the same question. Exact-match or word-overlap metrics penalize valid paraphrasing.
2. **Failure modes are subtle.** A response can be fluent, confident, and completely wrong (hallucination).
   Traditional metrics don't distinguish "wrong" from "wrong but sounds right."
3. **Evaluation criteria are task-specific.** A summarization system cares about faithfulness and conciseness.
   A RAG chatbot cares about faithfulness to retrieved context and answer relevancy. A code assistant cares
   about whether the code runs. One metric never covers every use case.

This is why production LLM evaluation uses a **stack of metrics**, not one number. Today we build the three
most commonly used checks: a traditional overlap metric (ROUGE), a faithfulness check, and an answer relevancy
check.


## 2. Traditional Metric: ROUGE (and Its Limits)

ROUGE measures word/n-gram overlap between a generated response and a reference response. It's fast, requires
no API calls, and is still used as a first-pass sanity check in production pipelines. But it penalizes correct
answers that are phrased differently from the reference, which is common with LLMs.


In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

reference = "The Eiffel Tower is located in Paris, France, and was completed in 1889."

responses = {
    "Close paraphrase": "The Eiffel Tower is in Paris, France. It was finished in 1889.",
    "Correct but different wording": "Gustave Eiffel's tower stands in the French capital and opened its doors in 1889.",
    "Vague / incomplete": "The Eiffel Tower is a famous landmark somewhere in Europe.",
}

rows = []
for label, resp in responses.items():
    scores = scorer.score(reference, resp)
    rows.append({
        "response_type": label,
        "response": resp,
        "rouge1_f1": round(scores['rouge1'].fmeasure, 3),
        "rougeL_f1": round(scores['rougeL'].fmeasure, 3),
    })

rouge_df = pd.DataFrame(rows)
rouge_df

**Observation:** notice that "Correct but different wording" scores lower on ROUGE than the close paraphrase,
even though it's factually just as correct. This is the exact limitation that pushed the field toward
semantic and LLM-based evaluation methods.


## 3. Faithfulness: Is the Answer Grounded in the Context?

Faithfulness (also called "groundedness") is the single most important metric for any RAG system. It answers
one question: **does every claim in the generated answer actually come from the retrieved context, or did the
model make something up?**

The Ragas library computes this by breaking the answer into individual claims and checking each one against
the context with an LLM judge. Here we build a lightweight proxy using TF-IDF cosine similarity between the
answer and the context, which runs with zero API calls and illustrates the same idea: a faithful answer should
overlap heavily with its source context, an unfaithful one won't.


In [ ]:
def text_similarity(a, b):
    vectorizer = TfidfVectorizer().fit([a, b])
    tfidf = vectorizer.transform([a, b])
    return cosine_similarity(tfidf[0], tfidf[1])[0][0]

context = (
    "The Eiffel Tower is a wrought-iron lattice tower in Paris, France. "
    "It was designed by engineer Gustave Eiffel and completed in 1889 for the World's Fair."
)

answers = {
    "Faithful": "The Eiffel Tower was designed by Gustave Eiffel and completed in 1889.",
    "Partially faithful": "The Eiffel Tower is a famous iron tower in Paris, built for an exhibition.",
    "Hallucinated": "The Eiffel Tower was built in 1920 and designed by Leonardo da Vinci.",
}

print(f"{'Answer type':<22}{'Faithfulness score':<20}")
for label, ans in answers.items():
    score = text_similarity(context, ans)
    print(f"{label:<22}{score:.3f}")

**What to look for:** the hallucinated answer scores noticeably lower because it introduces facts
(the year 1920, "Leonardo da Vinci") that don't appear anywhere in the context. In a production pipeline, a
faithfulness score below a set threshold (commonly 0.5-0.7 depending on the domain) gets flagged for review
or blocks the response entirely.

**Important limitation to be honest about:** TF-IDF overlap is a proxy, not a real faithfulness check. It can
be fooled by an answer that reuses the same words as the context but rearranges facts incorrectly. Production
systems (Ragas, and Chip Huyen's book both cover this) use an LLM to break the answer into atomic claims and
verify each one individually against the context. We build that pattern in Section 5.


## 4. Answer Relevancy: Does the Answer Address the Question?

A response can be perfectly faithful to the context and still be a bad answer if it doesn't address what was
asked. Answer relevancy checks the answer against the **question**, not the context.


In [ ]:
question = "When was the Eiffel Tower completed?"

candidate_answers = {
    "Directly relevant": "The Eiffel Tower was completed in 1889.",
    "Related but off-target": "Paris is the capital of France and a major tourist destination.",
    "Completely irrelevant": "The stock market closed higher today on strong earnings.",
}

print(f"{'Answer type':<24}{'Relevancy score':<18}")
for label, ans in candidate_answers.items():
    score = text_similarity(question, ans)
    print(f"{label:<24}{score:.3f}")

**Production note:** faithfulness and relevancy are measured independently and both matter. An answer can
be highly faithful (every word pulled straight from context) but low relevancy (answers a different question
than the one asked). You need both checks, not one.


## 5. The LLM-as-Judge Pattern

The proxies above are useful for building intuition and for fast, free, offline checks. But the actual
production standard, the pattern used by Ragas, OpenAI evals, and described extensively in Chip Huyen's
*AI Engineering*, is to use a strong LLM as the judge itself. You send it the question, the context, and the
generated answer, and ask it to score specific dimensions with reasoning.

Below is the exact prompt template used for this pattern, plus a function that calls it if you have an API key
configured, and falls back to a clear message if you don't. This is intentional: you should see the real
pattern even before you have API access, which is what Day 39 covers.


In [ ]:
def build_judge_prompt(question, context, answer):
    return f'''You are evaluating an AI assistant's answer for a RAG system.

Question: {question}
Context provided to the assistant: {context}
Assistant's answer: {answer}

Rate the answer from 1 (worst) to 5 (best) on each dimension:
1. Faithfulness: Is every claim in the answer supported by the context?
2. Relevancy: Does the answer directly address the question?
3. Completeness: Does the answer cover what the context allows it to cover?

Respond ONLY in JSON with this exact structure:
{{"faithfulness": <1-5>, "relevancy": <1-5>, "completeness": <1-5>, "reasoning": "<one sentence>"}}'''


def llm_judge(question, context, answer, api_key=None):
    '''
    Calls an LLM to score a RAG answer. Requires an API key (OpenAI, Anthropic, etc.)
    Falls back to printing the prompt if no key is provided, so you can see exactly
    what production evaluation pipelines send to the judge model.
    '''
    prompt = build_judge_prompt(question, context, answer)

    if api_key is None:
        print("No API key provided. Here is the exact prompt a judge LLM would receive:\n")
        print(prompt)
        return None

    # Example wiring for the OpenAI API (uncomment and adapt once you have a key, Day 39 covers this in depth):
    # from openai import OpenAI
    # client = OpenAI(api_key=api_key)
    # response = client.chat.completions.create(
    #     model="gpt-4o-mini",
    #     messages=[{"role": "user", "content": prompt}],
    #     response_format={"type": "json_object"}
    # )
    # return response.choices[0].message.content

    return "API call wiring goes here — see Day 39."


# Demo run without an API key
llm_judge(
    question="When was the Eiffel Tower completed?",
    context="The Eiffel Tower was designed by Gustave Eiffel and completed in 1889.",
    answer="The Eiffel Tower was completed in 1889."
)

**Why this pattern matters for interviews and production work:** LLM-as-judge scales far better than
human review and captures nuance that overlap-based metrics miss entirely. Its known weaknesses, covered in
both the Ragas docs and Chip Huyen's book, are cost (every evaluation is an API call), latency, and judge bias
(the judge model can favor answers that are stylistically similar to its own outputs). Production teams
typically combine cheap proxy metrics like the ones in Sections 3-4 for every request, and reserve LLM-as-judge
for sampled batches or offline evaluation runs.


## 6. Practice Exercise: Evaluate 5 RAG Answers

Below are 5 question-context-answer triples from a mock product support RAG system. Two of them contain a
hallucination. Run the faithfulness and relevancy checks on all 5 and identify which two are the hallucinated
ones **before** revealing the answer key.


In [ ]:
qa_pairs = [
    {
        "id": 1,
        "question": "What is the return window for electronics?",
        "context": "Electronics can be returned within 30 days of purchase with the original receipt.",
        "answer": "You can return electronics within 30 days of purchase if you have the receipt.",
    },
    {
        "id": 2,
        "question": "Does the premium plan include priority support?",
        "context": "The premium plan includes priority support with a 2-hour response time guarantee.",
        "answer": "Yes, the premium plan includes priority support with a 2-hour response guarantee, plus a free annual hardware upgrade.",
    },
    {
        "id": 3,
        "question": "What payment methods are accepted?",
        "context": "We accept Visa, Mastercard, and PayPal. Cryptocurrency is not supported.",
        "answer": "We accept Visa, Mastercard, and PayPal.",
    },
    {
        "id": 4,
        "question": "How long does standard shipping take?",
        "context": "Standard shipping takes 5-7 business days within the country.",
        "answer": "Standard shipping is typically same-day delivery in most major cities.",
    },
    {
        "id": 5,
        "question": "Can I cancel my subscription anytime?",
        "context": "Subscriptions can be cancelled anytime from the account settings page, effective at the end of the billing cycle.",
        "answer": "Yes, you can cancel anytime from account settings, and it takes effect at the end of your billing cycle.",
    },
]

results = []
for pair in qa_pairs:
    faithfulness = text_similarity(pair["context"], pair["answer"])
    relevancy = text_similarity(pair["question"], pair["answer"])
    results.append({
        "id": pair["id"],
        "faithfulness": round(faithfulness, 3),
        "relevancy": round(relevancy, 3),
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Flag any answer with faithfulness below a threshold for review
THRESHOLD = 0.45
flagged = results_df[results_df["faithfulness"] < THRESHOLD]
print("Flagged for possible hallucination:")
flagged

### Answer key

Run the cell below only after you've made your own guess.


In [ ]:
answer_key = '''
Q2 is the hallucination that's easy to miss: it adds a claim ("free annual hardware upgrade")
that never appears in the context. This is the most common and most dangerous type of hallucination,
it's mixed in with mostly-correct information, which makes it harder to catch by skimming.

Q4 is a clear hallucination: the context says 5-7 business days, the answer claims same-day delivery
in most major cities. This directly contradicts the source.

Q1, Q3, and Q5 are faithful, every claim in the answer traces back to the context.

This is exactly why faithfulness has to be checked automatically at scale. A single wrong number
or added claim, buried inside an otherwise accurate answer, is very easy for a human reviewer to miss
but has real consequences in a production support system.
'''
print(answer_key)

## 7. Reflection

Before moving to Day 39, answer this for yourself:

If you were building the evaluation pipeline for a production RAG system today, which metric would you run on
**every single request** (cheap, fast, automatic) versus which would you reserve for a **sampled offline batch**
(expensive, slower, more accurate)? Write your reasoning in the markdown cell below.

*(Your answer here)*


## Self-Check Before Day 39

You're ready to move on if you can answer these without looking back:

1. Why do traditional overlap metrics like ROUGE penalize valid LLM outputs?
2. What's the difference between faithfulness and answer relevancy, and why do you need both?
3. What specific weakness does LLM-as-judge have that proxy metrics like TF-IDF similarity don't?
4. In the practice exercise, what made the Q2 hallucination harder to catch than the Q4 hallucination?

**Full series repo:** github.com/VaishnaviJagtap18/42-days-aiml-challenge

Day 39 tomorrow: Building with APIs (OpenAI, Anthropic, Google). This is where the `llm_judge` function above
gets wired up for real.
